In [ ]:
###
    # This code vertically interpolates soil temperature and moisture data from each CMIP6 model to the vertical resolution of the moisture-driven model (based on CoLM).
    # 1. Read monthly CMIP6 soil temperature (tsl) and soil water (mrsll & mrsfl) data for the period 1982–2014.
    # 2. For soil water, convert the unit to density (kg/m³) prior to vertical interpolation, then interpolate the water density.
    # 3. Mask out non-permafrost regions, retaining only permafrost grid cells.
    # Output: "../Data/inter_data/"
###

In [1]:
import numpy as np
import xarray as xr
from scipy import interpolate
import glob
from scipy.interpolate import griddata
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 1. Prepare CoLM vertical layers
# Node depths (target depths for interpolation)
nl_soil = 15           # upper bound of array
x_array = np.array([1,2,3,4,5,6,7,8,9,10,11,12,13,14,15])
colm_node_depths = 0.025*(np.exp(0.5*(x_array-0.5))-1)  # node depth [m]
print(f"\nCoLM node_depths (m):")

for i, colm_node_depth in enumerate(colm_node_depths):
    print(f"  Layer{i+1:2d}: {colm_node_depth:.4f} m")
dz_soisno = np.zeros(nl_soil)    # soil layer thickness [m]
dz_soisno[0] = 0.5*(colm_node_depths[1]+colm_node_depths[0])
for d in range(1,nl_soil-1):
    dz_soisno[d] = 0.5*(colm_node_depths[d+1]-colm_node_depths[d-1])
dz_soisno[nl_soil-1] = colm_node_depths[nl_soil-1] - colm_node_depths[nl_soil-2]

# Interface depths (for calculating layer thickness)
colm_interface_depths = np.zeros(nl_soil+1)  # interface depth [m]
colm_interface_depths[0]=0.
for d in range(1,nl_soil+1):
    colm_interface_depths[d]=colm_interface_depths[d-1]+dz_soisno[d-1]
    
# 2. Calculate CoLM layer thicknesses
colm_layer_thickness = colm_interface_depths[1:] - colm_interface_depths[:-1]
print(f"\nCoLM layer thicknesses (m):")
for i, thickness in enumerate(colm_layer_thickness):
    print(f"  Layer{i+1:2d}: {thickness:.4f} m")

# 3. Vertical interpolation function (for intensive variables)
def vertical_interpolate_intensity(data_intensity, depths_original, depths_target, fill_value=np.nan):
    """
    Interpolate intensive variable data from original depths to target depths
    
    Parameters:
    data_intensity: original depth-intensive variable data (n_depth, ...) units: kg/m³ or tsl (units: K)
    depths_original: original depth node positions (m)
    depths_target: target depth node positions (m)
    fill_value: fill value
    
    Returns:
    Interpolated intensive variable data
    """
    # Get data shape
    original_shape = data_intensity.shape
    target_shape = (len(depths_target),) + original_shape[1:]
    
    # Initialize result array
    result = np.full(target_shape, fill_value, dtype=np.float32)
    
    # Create mask to identify valid data
    valid_mask = data_intensity != fill_value
    
    # Interpolate for each spatial location
    for lat_idx in range(original_shape[1]):
        for lon_idx in range(original_shape[2]):
            # Extract valid depth data at this location
            valid_indices = valid_mask[:, lat_idx, lon_idx]
            
            if np.any(valid_indices):
                # Get valid depth values
                valid_depths = depths_original[valid_indices]
                valid_values = data_intensity[valid_indices, lat_idx, lon_idx]
                
                # Ensure depths are increasing
                sort_idx = np.argsort(valid_depths)
                valid_depths_sorted = valid_depths[sort_idx]
                valid_values_sorted = valid_values[sort_idx]
                
                # Remove duplicate depth values
                unique_depths, unique_indices = np.unique(valid_depths_sorted, return_index=True)
                valid_depths_unique = valid_depths_sorted[unique_indices]
                valid_values_unique = valid_values_sorted[unique_indices]
                
                if len(valid_depths_unique) > 1:
                    # Use linear interpolation
                    interp_func = interpolate.interp1d(
                        valid_depths_unique, 
                        valid_values_unique,
                        kind='linear',
                        bounds_error=False,
                        fill_value=(valid_values_unique[0], valid_values_unique[-1])
                    )
                    
                    # Interpolate to target depths
                    interp_values = interp_func(depths_target)
                    result[:, lat_idx, lon_idx] = interp_values
                elif len(valid_depths_unique) == 1:
                    # If only one valid depth, use that value for all depths
                    result[:, lat_idx, lon_idx] = valid_values_unique[0]
    
    return result


CoLM node_depths (m):
  Layer 1: 0.0071 m
  Layer 2: 0.0279 m
  Layer 3: 0.0623 m
  Layer 4: 0.1189 m
  Layer 5: 0.2122 m
  Layer 6: 0.3661 m
  Layer 7: 0.6198 m
  Layer 8: 1.0380 m
  Layer 9: 1.7276 m
  Layer10: 2.8646 m
  Layer11: 4.7392 m
  Layer12: 7.8298 m
  Layer13: 12.9253 m
  Layer14: 21.3265 m
  Layer15: 35.1776 m

CoLM layer thicknesses (m):
  Layer 1: 0.0175 m
  Layer 2: 0.0276 m
  Layer 3: 0.0455 m
  Layer 4: 0.0750 m
  Layer 5: 0.1236 m
  Layer 6: 0.2038 m
  Layer 7: 0.3360 m
  Layer 8: 0.5539 m
  Layer 9: 0.9133 m
  Layer10: 1.5058 m
  Layer11: 2.4826 m
  Layer12: 4.0931 m
  Layer13: 6.7484 m
  Layer14: 11.1262 m
  Layer15: 13.8512 m


In [ ]:
models = [
    "CESM2", "CESM2-FV2", "CESM2-WACCM",
          "CNRM-CM6-1", "CNRM-ESM2-1",
          "GFDL-ESM4","NorESM2-LM", "NorESM2-MM", "TaiESM1"
          ]
# Loop through each model
for i, model in enumerate(models):
    # Read NetCDF data
    filepath = glob.glob(f"/home/jidy/Data/tas-tsl-merge/tsl_{model}_r1i1p1f1_185001-210012.nc")[0]
    ds = xr.open_dataset(filepath)
    
    # 4. Extract data for the period 1982-2014
    try:
        years = np.array([dt.year for dt in ds.time.values])
        year_mask = (years >= 1982) & (years <= 2014)
        ds = ds.sel(lat=slice(45,90), lon=slice(0,180))
        ds_subset = ds.isel(time=year_mask)
    except:
        ds = ds.sel(lat=slice(45,90), lon=slice(0,180))
        ds_subset = ds.sel(time=slice('1982-01-01', '2014-12-31'))
        
    # Calculate multi-year mean
    tsl_data =  ds_subset['tsl'].mean(dim='time')

    # 5. Get original depth and depth boundary data
    # Original depths (node depths)
    try:
        original_depths = ds['depth'].values
        original_depth_bnds = ds['depth_bnds'].values
        # 6. Calculate original layer thicknesses (m)
        original_layer_thickness = original_depth_bnds[:,1]-original_depth_bnds[:,0]    
    except:
        try:
            original_depths = ds['sdepth'].values
            original_depth_bnds = ds['sdepth_bounds'].values
            # 6. Calculate original layer thicknesses (m)
            original_layer_thickness = original_depth_bnds[:,1]-original_depth_bnds[:,0]
        except:
            original_depths = ds['depth'].values
            zhalf_soil = ds['zhalf_soil'].values
            # 6. Calculate original layer thicknesses (m)
            original_layer_thickness = np.diff(zhalf_soil) 
    
    # 8. Perform vertical interpolation for each time step    
    lat_len = len(tsl_data.lat)
    lon_len = len(tsl_data.lon)
    colm_depth_len = len(colm_node_depths)
    fill_value = np.nan
    
    # Initialize result array (intensive variable)
    interpolated_intensity = np.full((colm_depth_len, lat_len, lon_len), 
                                     fill_value, dtype=np.float32)
        
    # Perform vertical interpolation (on intensive variable)
    interpolated_intensity = vertical_interpolate_intensity(
        tsl_data, 
        original_depths, 
        colm_node_depths,
        fill_value=fill_value
        )
           
    # 10. Create new dataset
    # Create new depth coordinate
    new_depth_coord = xr.DataArray(
        colm_node_depths,
        dims=['depth'],
        attrs={
            'long_name': 'CoLM soil layer node depths',
            'units': 'm',
            'positive': 'down',
            'standard_name': 'depth',
            'bounds': 'depth_bnds'
        }
    )
    
    # Create new dataset (storing extensive variable)
    ds_new = xr.Dataset(
        {
            'tsl': xr.DataArray(
                data=interpolated_intensity,
                dims=['depth', 'lat', 'lon'],
                coords={
                    'depth': new_depth_coord,
                    'lat': tsl_data.lat,
                    'lon': tsl_data.lon
                },
                attrs={
                    '_FillValue': fill_value,
                    'long_name': 'Soil Temperature',
                    'units': 'K',
                    'standard_name': 'temperature_of_soil_layer',
                    'description': 'Vertically integrated soil temperature in soil layers',
                    'processing_history': 'Converted from extensive to intensive, vertically interpolated to CoLM layers, then converted back to extensive',
                    'interpolation_method': 'linear interpolation',
                    'original_depths': str(original_depths.tolist()),
                    'CoLM_depths': str(colm_node_depths.tolist())
                }
            )
        }
    )
    
    # Add depth boundary information
    depth_bnds_data = np.column_stack([colm_interface_depths[:-1], colm_interface_depths[1:]])
    ds_new['depth_bnds'] = xr.DataArray(
        data=depth_bnds_data,
        dims=['depth', 'bnds'],
        attrs={
            'long_name': 'CoLM soil layer depth boundaries',
            'units': 'm',
            'comment': 'Interface depths of CoLM soil layers',
            'positive': 'down'
        }
    )
    
    # Add layer thickness information
    ds_new['layer_thickness'] = xr.DataArray(
        data=colm_layer_thickness,
        dims=['depth'],
        attrs={
            'long_name': 'CoLM soil layer thickness',
            'units': 'm',
            'comment': 'Thickness of each soil layer'
        }
    )
    
    # 11. Save results
    output_file = f'../Data/inter_data/tsl_{model}_historical_1982-2014_CoLM_vertical.nc'
    encoding = {
        'tsl': {
            #'zlib': True,
            #'complevel': 5,
            #'_FillValue': fill_value,
            'dtype': 'float32'
        }
    }
    ds_new.to_netcdf(output_file, encoding=encoding)

In [ ]:
# Data folder path
data_folder = "./Data/inter_data/"
probability_file = "../Data/probability_lt_threshold_320.nc"

# Read probability threshold data
ds_prob = xr.open_dataset(probability_file)
ds_prob = ds_prob.sel(lon=slice(0, 180))
mask = ds_prob['probability_lt_threshold'] > 0
lon_prob, lat_prob = np.meshgrid(ds_prob.lon.values, ds_prob.lat.values)

# Loop through each model
for model in models:
    filepath = glob.glob(data_folder+f"tsl_{model}_historical_1982-2014_CoLM_vertical.nc")[0]
    
    # Read soil temperature data
    ds = xr.open_dataset(filepath)
    tsl = ds['tsl']
    
    # Create grid for soil data
    lon_soil, lat_soil = np.meshgrid(tsl.lon.values, tsl.lat.values)
    
    # Interpolate mask to soil data grid
    mask_interp = griddata((lon_prob.ravel(), lat_prob.ravel()), 
                            mask.values.ravel(),
                            (lon_soil.ravel(), lat_soil.ravel()), 
                            method='nearest')
    mask_interp = mask_interp.reshape(lon_soil.shape).astype(bool)
    
    # Create mask DataArray
    mask_da = xr.DataArray(mask_interp, 
                            coords={'lat': tsl.lat, 'lon': tsl.lon},
                            dims=['lat', 'lon'])
    
    # Apply mask to filter data
    tsl_masked = tsl.where(mask_da, drop=False)
    
    # ====== Create new Dataset and save ======
    
    # 1. Create new Dataset
    ds_new = xr.Dataset()
    
    # 2. Add multi-year mean data variable
    ds_new['tsl'] = tsl_masked
    
    # 3. Add coordinate variables
    ds_new = ds_new.assign_coords({
        'depth': tsl.depth,
        'lat': tsl.lat,
        'lon': tsl.lon
    })
    
    # 4. Add variable attributes
    ds_new['tsl'].attrs = {
        'long_name': 'Mean Soil Temperature',
        'units': tsl.attrs.get('units', 'unknown'),
        'description': 'Multi-year (1982-2014) mean of soil temperature after regional mask applied'
    }
    
    # 5. Add global attributes
    ds_new.attrs = {
        'title': f'Mean Soil Temperature for {model} (1982-2014)',
        'model': model,
        'data_source': 'CMIP6 historical simulation',
        'region': 'CoLM grid with regional mask applied',
        'processing': 'Masked by probability threshold > 0, then averaged over time dimension',
        'created_by': 'Soil temperature analysis script',
        'creation_date': np.datetime64('today').astype(str)
    }
    
    # 6. Save to file
    output_filename = f"tsl_{model}_historical_1982-2014_CoLM_vertical_pf_region.nc"
    output_path = data_folder + output_filename
    
    # Save file
    ds_new.to_netcdf(output_path)

In [ ]:
# Loop through each model
for i, model in enumerate(models):
    # Read NetCDF data
    filepath = glob.glob(f"../Data/cmip6_mrsll/mrsll_Emon_{model}_*.nc")[0]
    ds = xr.open_dataset(filepath)
    
    # 4. Extract data for the period 1982-2014
    try:
        years = np.array([dt.year for dt in ds.time.values])
        year_mask = (years >= 1982) & (years <= 2014)
        ds = ds.sel(lat=slice(45,90), lon=slice(0,190))
        ds_subset = ds.isel(time=year_mask)
    except:
        ds = ds.sel(lat=slice(45,90), lon=slice(0,190))
        ds_subset = ds.sel(time=slice('1982-01-01', '2014-12-31'))
    
    # 5. Get original depth and depth boundary data
    # Original depths (node depths)
    try:
        original_depths = ds['depth'].values
        original_depth_bnds = ds['depth_bnds'].values
        # 6. Calculate original layer thicknesses (m)
        original_layer_thickness = original_depth_bnds[:,1]-original_depth_bnds[:,0]    
    except:
        try:
            original_depths = ds['sdepth'].values
            original_depth_bnds = ds['sdepth_bounds'].values
            # 6. Calculate original layer thicknesses (m)
            original_layer_thickness = original_depth_bnds[:,1]-original_depth_bnds[:,0]
        except:
            original_depths = ds['depth'].values
            zhalf_soil = ds['zhalf_soil'].values
            # 6. Calculate original layer thicknesses (m)
            original_layer_thickness = np.diff(zhalf_soil)
    
    # 7. Convert extensive quantity (kg/m²) to intensive quantity (kg/m³)
    original_data = ds_subset.mrsll.values  # shape: (time, depth, lat, lon)
    
    # Create intensive quantity array
    intensity_data = np.full_like(original_data, np.nan, dtype=np.float32)
    
    # Convert for each depth layer
    for d_idx in range(len(original_depths)):
        thickness = original_layer_thickness[d_idx]
        # Intensive = Extensive / layer thickness
        # Use numpy's nan handling
        layer_data = original_data[:, d_idx, :, :]
        valid_mask = ~np.isnan(layer_data)
        if np.any(valid_mask):
            intensity_data[:, d_idx, :, :][valid_mask] = layer_data[valid_mask] / thickness
    
    # 8. Perform vertical interpolation for each time step    
    time_len = len(ds_subset.time)
    lat_len = len(ds_subset.lat)
    lon_len = len(ds_subset.lon)
    colm_depth_len = len(colm_node_depths)
    fill_value = np.nan
    
    # Initialize result array (intensive quantity)
    interpolated_intensity = np.full((time_len, colm_depth_len, lat_len, lon_len), 
                                     fill_value, dtype=np.float32)
    
    # Process each time step
    for t_idx in range(time_len):
        
        # Extract intensive quantity data for current time step
        current_intensity = intensity_data[t_idx]
        
        # Perform vertical interpolation (on intensive quantity)
        interpolated_intensity[t_idx] = vertical_interpolate_intensity(
            current_intensity, 
            original_depths, 
            colm_node_depths,
            fill_value=fill_value
        )
        
    # 9. Convert interpolated intensive quantity back to extensive quantity
    # Initialize extensive quantity array
    interpolated_extensive = np.full_like(interpolated_intensity, fill_value, dtype=np.float32)
    
    # Convert for each CoLM layer
    for d_idx in range(colm_depth_len):
        thickness = colm_layer_thickness[d_idx]
        # Extensive = Intensive × layer thickness
        layer_intensity = interpolated_intensity[:, d_idx, :, :]
        valid_mask = layer_intensity != fill_value
        interpolated_extensive[:, d_idx, :, :][valid_mask] = layer_intensity[valid_mask] * thickness
        
    # 10. Create new dataset
    # Create new depth coordinate
    new_depth_coord = xr.DataArray(
        colm_node_depths,
        dims=['depth'],
        attrs={
            'long_name': 'CoLM soil layer node depths',
            'units': 'm',
            'positive': 'down',
            'standard_name': 'depth',
            'bounds': 'depth_bnds'
        }
    )
    
    # Create new dataset (storing extensive quantity)
    ds_new = xr.Dataset(
        {
            'mrsll': xr.DataArray(
                data=interpolated_extensive,
                dims=['time', 'depth', 'lat', 'lon'],
                coords={
                    'time': ds_subset.time,
                    'depth': new_depth_coord,
                    'lat': ds_subset.lat,
                    'lon': ds_subset.lon
                },
                attrs={
                    '_FillValue': fill_value,
                    'long_name': 'Liquid Water Content of Soil Layer',
                    'units': 'kg m-2',
                    'standard_name': 'liquid_water_content_of_soil_layer',
                    'description': 'Vertically integrated liquid water content in soil layers',
                    'processing_history': 'Converted from extensive to intensive, vertically interpolated to CoLM layers, then converted back to extensive',
                    'original_units': 'kg m-2',
                    'interpolation_method': 'linear interpolation on intensive quantity (kg/m³)',
                    'original_depths': str(original_depths.tolist()),
                    'CoLM_depths': str(colm_node_depths.tolist())
                }
            )
        }
    )
    
    # Add depth boundary information
    depth_bnds_data = np.column_stack([colm_interface_depths[:-1], colm_interface_depths[1:]])
    ds_new['depth_bnds'] = xr.DataArray(
        data=depth_bnds_data,
        dims=['depth', 'bnds'],
        attrs={
            'long_name': 'CoLM soil layer depth boundaries',
            'units': 'm',
            'comment': 'Interface depths of CoLM soil layers',
            'positive': 'down'
        }
    )
    
    # Add layer thickness information
    ds_new['layer_thickness'] = xr.DataArray(
        data=colm_layer_thickness,
        dims=['depth'],
        attrs={
            'long_name': 'CoLM soil layer thickness',
            'units': 'm',
            'comment': 'Thickness of each soil layer'
        }
    )
    
    # 11. Save results
    output_file = f'../Data/inter_data/mrsll_{model}_historical_1982-2014_CoLM_vertical.nc'
    encoding = {
        'mrsll': {
            #'zlib': True,
            #'complevel': 5,
            #'_FillValue': fill_value,
            'dtype': 'float32'
        }
    }
    ds_new.to_netcdf(output_file, encoding=encoding)

In [ ]:
# Loop through each model
for i, model in enumerate(models):
    # Read NetCDF data
    filepath = glob.glob(f"../Data/cmip6_mrsfl/mrsfl_Emon_{model}_*.nc")[0]
    ds = xr.open_dataset(filepath)
    
    # 4. Extract data for the period 1982-2014
    try:
        years = np.array([dt.year for dt in ds.time.values])
        year_mask = (years >= 1982) & (years <= 2014)
        ds = ds.sel(lat=slice(45,90), lon=slice(0,190))
        ds_subset = ds.isel(time=year_mask)
    except:
        ds = ds.sel(lat=slice(45,90), lon=slice(0,190))
        ds_subset = ds.sel(time=slice('1982-01-01', '2014-12-31'))
    
    # 5. Get original depth and depth boundary data
    # Original depths (node depths)
    try:
        original_depths = ds['depth'].values
        original_depth_bnds = ds['depth_bnds'].values
        # 6. Calculate original layer thicknesses (m)
        original_layer_thickness = original_depth_bnds[:,1]-original_depth_bnds[:,0]    
    except:
        try:
            original_depths = ds['sdepth'].values
            original_depth_bnds = ds['sdepth_bounds'].values
            # 6. Calculate original layer thicknesses (m)
            original_layer_thickness = original_depth_bnds[:,1]-original_depth_bnds[:,0]
        except:
            original_depths = ds['depth'].values
            zhalf_soil = ds['zhalf_soil'].values
            # 6. Calculate original layer thicknesses (m)
            original_layer_thickness = np.diff(zhalf_soil)
    
    # 7. Convert extensive quantity (kg/m²) to intensive quantity (kg/m³)
    original_data = ds_subset.mrsfl.values  # shape: (time, depth, lat, lon)
    
    # Create intensive quantity array
    intensity_data = np.full_like(original_data, np.nan, dtype=np.float32)
    
    # Convert for each depth layer
    for d_idx in range(len(original_depths)):
        thickness = original_layer_thickness[d_idx]
        # Intensive = Extensive / layer thickness
        # Use numpy's nan handling
        layer_data = original_data[:, d_idx, :, :]
        valid_mask = ~np.isnan(layer_data)
        if np.any(valid_mask):
            intensity_data[:, d_idx, :, :][valid_mask] = layer_data[valid_mask] / thickness
    
    # 8. Perform vertical interpolation for each time step
    time_len = len(ds_subset.time)
    lat_len = len(ds_subset.lat)
    lon_len = len(ds_subset.lon)
    colm_depth_len = len(colm_node_depths)
    fill_value = np.nan
    
    # Initialize result array (intensive quantity)
    interpolated_intensity = np.full((time_len, colm_depth_len, lat_len, lon_len), 
                                     fill_value, dtype=np.float32)
    
    # Process each time step
    for t_idx in range(time_len):
        
        # Extract intensive quantity data for current time step
        current_intensity = intensity_data[t_idx]
        
        # Perform vertical interpolation (on intensive quantity)
        interpolated_intensity[t_idx] = vertical_interpolate_intensity(
            current_intensity, 
            original_depths, 
            colm_node_depths,
            fill_value=fill_value
        )
    
    # 9. Convert interpolated intensive quantity back to extensive quantity
    # Initialize extensive quantity array
    interpolated_extensive = np.full_like(interpolated_intensity, fill_value, dtype=np.float32)
    
    # Convert for each CoLM layer
    for d_idx in range(colm_depth_len):
        thickness = colm_layer_thickness[d_idx]
        # Extensive = Intensive × layer thickness
        layer_intensity = interpolated_intensity[:, d_idx, :, :]
        valid_mask = layer_intensity != fill_value
        interpolated_extensive[:, d_idx, :, :][valid_mask] = layer_intensity[valid_mask] * thickness
        
    # 10. Create new dataset
    # Create new depth coordinate
    new_depth_coord = xr.DataArray(
        colm_node_depths,
        dims=['depth'],
        attrs={
            'long_name': 'CoLM soil layer node depths',
            'units': 'm',
            'positive': 'down',
            'standard_name': 'depth',
            'bounds': 'depth_bnds'
        }
    )
    
    # Create new dataset (storing extensive quantity)
    ds_new = xr.Dataset(
        {
            'mrsfl': xr.DataArray(
                data=interpolated_extensive,
                dims=['time', 'depth', 'lat', 'lon'],
                coords={
                    'time': ds_subset.time,
                    'depth': new_depth_coord,
                    'lat': ds_subset.lat,
                    'lon': ds_subset.lon
                },
                attrs={
                    '_FillValue': fill_value,
                    'long_name': 'Frozen Water Content of Soil Layer',
                    'units': 'kg m-2',
                    'standard_name': 'frozen_water_content_of_soil_layer',
                    'description': 'Vertically integrated frozen water content in soil layers',
                    'processing_history': 'Converted from extensive to intensive, vertically interpolated to CoLM layers, then converted back to extensive',
                    'original_units': 'kg m-2',
                    'interpolation_method': 'linear interpolation on intensive quantity (kg/m³)',
                    'original_depths': str(original_depths.tolist()),
                    'CoLM_depths': str(colm_node_depths.tolist())
                }
            )
        }
    )
    
    # Add depth boundary information
    depth_bnds_data = np.column_stack([colm_interface_depths[:-1], colm_interface_depths[1:]])
    ds_new['depth_bnds'] = xr.DataArray(
        data=depth_bnds_data,
        dims=['depth', 'bnds'],
        attrs={
            'long_name': 'CoLM soil layer depth boundaries',
            'units': 'm',
            'comment': 'Interface depths of CoLM soil layers',
            'positive': 'down'
        }
    )
    
    # Add layer thickness information
    ds_new['layer_thickness'] = xr.DataArray(
        data=colm_layer_thickness,
        dims=['depth'],
        attrs={
            'long_name': 'CoLM soil layer thickness',
            'units': 'm',
            'comment': 'Thickness of each soil layer'
        }
    )
    
    # 11. Save results
    output_file = f'../Data/inter_data/mrsfl_{model}_historical_1982-2014_CoLM_vertical.nc'
    encoding = {
        'mrsfl': {
            #'zlib': True,
            #'complevel': 5,
            #'_FillValue': fill_value,
            'dtype': 'float32'
        }
    }
    ds_new.to_netcdf(output_file, encoding=encoding)

In [ ]:
# Loop through each model
for model in models:
    filepath = glob.glob(data_folder+f"mrsfl_{model}_historical_1982-2014_CoLM_vertical.nc")[0]
    
    # Read soil water content data
    ds = xr.open_dataset(filepath)
    mrsfl = ds['mrsfl']
    
    # Create grid for soil data
    lon_soil, lat_soil = np.meshgrid(mrsfl.lon.values, mrsfl.lat.values)
    
    # Interpolate mask to soil data grid
    mask_interp = griddata((lon_prob.ravel(), lat_prob.ravel()), 
                            mask.values.ravel(),
                            (lon_soil.ravel(), lat_soil.ravel()), 
                            method='nearest')
    mask_interp = mask_interp.reshape(lon_soil.shape).astype(bool)
    
    # Create mask DataArray
    mask_da = xr.DataArray(mask_interp, 
                            coords={'lat': mrsfl.lat, 'lon': mrsfl.lon},
                            dims=['lat', 'lon'])
    
    # Apply mask to filter data
    density_water_masked = mrsfl.where(mask_da, drop=False)
    
    # Calculate multi-year mean
    mean_yearly_range = density_water_masked.mean(dim='time')

    # ====== Create new Dataset and save ======
    
    # 1. Create new Dataset
    ds_new = xr.Dataset()
    
    # 2. Add multi-year mean data variable
    ds_new['mrsfl'] = mean_yearly_range
    
    # 3. Add coordinate variables
    ds_new = ds_new.assign_coords({
        'depth': mrsfl.depth,
        'lat': mrsfl.lat,
        'lon': mrsfl.lon
    })
    
    # 4. Add variable attributes
    ds_new['mrsfl'].attrs = {
        'long_name': 'Mean Soil Water Content',
        'units': mrsfl.attrs.get('units', 'unknown'),
        'description': 'Multi-year (1982-2014) mean of soil water content after regional mask applied'
    }
    
    # 5. Add global attributes
    ds_new.attrs = {
        'title': f'Mean Soil Water Content for {model} (1982-2014)',
        'model': model,
        'data_source': 'CMIP6 historical simulation',
        'region': 'CoLM grid with regional mask applied',
        'processing': 'Masked by probability threshold > 0, then averaged over time dimension',
        'created_by': 'Soil water analysis script',
        'creation_date': np.datetime64('today').astype(str)
    }
    
    # 6. Save to file
    output_filename = f"mrsfl_{model}_historical_1982-2014_CoLM_vertical_pf_region_15layers.nc"
    output_path = data_folder + output_filename
    
    # Save file
    ds_new.to_netcdf(output_path)

In [ ]:
# Loop through each model
for model in models:
    filepath = glob.glob(data_folder+f"mrsll_{model}_historical_1982-2014_CoLM_vertical.nc")[0]
    
    # Read soil water content data
    ds = xr.open_dataset(filepath)
    mrsll = ds['mrsll']
    
    # Create grid for soil data
    lon_soil, lat_soil = np.meshgrid(mrsll.lon.values, mrsll.lat.values)
    
    # Interpolate mask to soil data grid
    mask_interp = griddata((lon_prob.ravel(), lat_prob.ravel()), 
                            mask.values.ravel(),
                            (lon_soil.ravel(), lat_soil.ravel()), 
                            method='nearest')
    mask_interp = mask_interp.reshape(lon_soil.shape).astype(bool)
    
    # Create mask DataArray
    mask_da = xr.DataArray(mask_interp, 
                            coords={'lat': mrsll.lat, 'lon': mrsll.lon},
                            dims=['lat', 'lon'])
    
    # Apply mask to filter data
    density_water_masked = mrsll.where(mask_da, drop=False)
    
    # Calculate multi-year mean
    mean_yearly_range = density_water_masked.mean(dim='time')

    # ====== Create new Dataset and save ======
    
    # 1. Create new Dataset
    ds_new = xr.Dataset()
    
    # 2. Add multi-year mean data variable
    ds_new['mrsll'] = mean_yearly_range
    
    # 3. Add coordinate variables
    ds_new = ds_new.assign_coords({
        'depth': mrsll.depth,
        'lat': mrsll.lat,
        'lon': mrsll.lon
    })
    
    # 4. Add variable attributes
    ds_new['mrsll'].attrs = {
        'long_name': 'Mean Soil Water Content',
        'units': mrsll.attrs.get('units', 'unknown'),
        'description': 'Multi-year (1982-2014) mean of soil water content after regional mask applied'
    }
    
    # 5. Add global attributes
    ds_new.attrs = {
        'title': f'Mean Soil Water Content for {model} (1982-2014)',
        'model': model,
        'data_source': 'CMIP6 historical simulation',
        'region': 'CoLM grid with regional mask applied',
        'processing': 'Masked by probability threshold > 0, then averaged over time dimension',
        'created_by': 'Soil water analysis script',
        'creation_date': np.datetime64('today').astype(str)
    }
    
    # 6. Save to file
    output_filename = f"mrsll_{model}_historical_1982-2014_CoLM_vertical_pf_region_15layers.nc"
    output_path = data_folder + output_filename
    
    # Save file
    ds_new.to_netcdf(output_path)